# 03 — Backtest Results

Reference card for the `BarBacktest` output. Documents the execution and equity
columns it appends on top of the signal frame from notebook 02 — the per-trade
log is in notebook 04, the equity curves in notebook 05.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalTradePerformance       (iid bets)
      (bars)            (signal cols)   (bt_result)    └─  SignalAllocationPerformance (capital deployed)
                                            ▲
                                       this notebook
```

**What this notebook covers**

1. **Data → Signal → Backtest** — minimal end-to-end wiring to produce a
   `BarBacktestResult`.
2. **`bt_result.data` schema** — every execution column appended by the
   backtest: `cycle`, three return series (`return_mark_to_close`,
   `return_conservative`, `return_net`), `position_start` / `position_end`,
   three matching equity curves, and `trade_cycle_id`. Single-symbol slice
   copied to clipboard for Excel inspection.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Data → Signal → Backtest

In [ ]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)

df = bt_result.data
print(f"{df.shape[0]:,} bars  |  {df.index.get_level_values('symbol').nunique()} symbols")

## 2. bt_result Structure

`BarBacktest.run()` returns a `BarBacktestResult` whose `.data` is a `(symbol, timestamp)`
MultiIndex DataFrame. The original signal columns are preserved and the backtest engine
appends execution and equity columns.

| Column | Description |
|---|---|
| `cycle` | `None` (flat), `init` (entry bar), `long` (in trade), `exit` (exit bar) |
| `return_mark_to_close` | Fill-aware daily return — entry on open after `init`, exit on open after `exit` |
| `return_conservative` | Worst-case fill — entry on `init` high, exit on `exit` low |
| `return_net` | MTC return minus round-trip cost (`cost_bps / 10 000` on entry and exit bars) |
| `position_start` | 1 when capital is deployed at bar open |
| `position_end` | 1 when capital is deployed at bar close |
| `strategy_equity_mark_to_close` | Cumulative product of MTC returns (starts at 1.0) |
| `strategy_equity_conservative` | Cumulative product of conservative returns |
| `strategy_equity_net` | Cumulative product of net returns |
| `trade_cycle_id` | Integer ID incrementing on each new trade or flat period |

In [3]:
bt_result.data.columns

Index(['open', 'high', 'low', 'close', 'volume', 'ma', 'signal_close',
       'signal_open', 'trade_direction', 'turnover', 'enter', 'exit', 'cycle',
       'signal_age', 'return_mark_to_close', 'return_conservative',
       'return_net', 'position_start', 'position_end',
       'strategy_equity_mark_to_close', 'strategy_equity_conservative',
       'strategy_equity_net', 'trade_cycle_id'],
      dtype='str')

In [4]:
exec_cols = [
    "cycle", "return_mark_to_close", "return_conservative", "return_net",
    "position_start", "position_end",
    "strategy_equity_mark_to_close", "strategy_equity_conservative", "strategy_equity_net",
    "trade_cycle_id",
]
df[exec_cols].head(12)

cycle  return_mark_to_close  return_conservative  \
symbol  timestamp                                                     
BTC-USD 2022-01-01  None                   0.0                  0.0   
        2022-01-02  None                   0.0                  0.0   
        2022-01-03  None                   0.0                  0.0   
        2022-01-04  None                   0.0                  0.0   
        2022-01-05  None                   0.0                  0.0   
        2022-01-06  None                   0.0                  0.0   
        2022-01-07  None                   0.0                  0.0   
        2022-01-08  None                   0.0                  0.0   
        2022-01-09  None                   0.0                  0.0   
        2022-01-10  None                   0.0                  0.0   
        2022-01-11  None                   0.0                  0.0   
        2022-01-12  None                   0.0                  0.0   

                    return_net  position_start  position_end  \
symbol  timestamp                                              
BTC-USD 2022-01-01         0.0               0             0   
        2022-01-02         0.0               0             0   
        2022-01-03         0.0               0             0   
        2022-01-04         0.0               0             0   
        2022-01-05         0.0               0             0   
        2022-01-06         0.0               0             0   
        2022-01-07         0.0               0             0   
        2022-01-08         0.0               0             0   
        2022-01-09         0.0               0             0   
        2022-01-10         0.0               0             0   
        2022-01-11         0.0               0             0   
        2022-01-12         0.0               0             0   

                    strategy_equity_mark_to_close  \
symbol  timestamp                                   
BTC-USD 2022-01-01                            1.0   
        2022-01-02                            1.0   
        2022-01-03                            1.0   
        2022-01-04                            1.0   
        2022-01-05                            1.0   
        2022-01-06                            1.0   
        2022-01-07                            1.0   
        2022-01-08                            1.0   
        2022-01-09                            1.0   
        2022-01-10                            1.0   
        2022-01-11                            1.0   
        2022-01-12                            1.0   

                    strategy_equity_conservative  strategy_equity_net  \
symbol  timestamp                                                       
BTC-USD 2022-01-01                           1.0                  1.0   
        2022-01-02                           1.0                  1.0   
        2022-01-03                           1.0                  1.0   
        2022-01-04                           1.0                  1.0   
        2022-01-05                           1.0                  1.0   
        2022-01-06                           1.0                  1.0   
        2022-01-07                           1.0                  1.0   
        2022-01-08                           1.0                  1.0   
        2022-01-09                           1.0                  1.0   
        2022-01-10                           1.0                  1.0   
        2022-01-11                           1.0                  1.0   
        2022-01-12                           1.0                  1.0   

                    trade_cycle_id  
symbol  timestamp                   
BTC-USD 2022-01-01               1  
        2022-01-02               1  
        2022-01-03               1  
        2022-01-04               1  
        2022-01-05               1  
        2022-01-06               1  
        2022-01-07               1  
        2022-01-08          

In [5]:
sym = "BTC-USD"
bt_result.data.query("symbol == @sym").unstack('symbol').to_clipboard()